In [1]:
# Fabric Notebook Source — Silver Layer Pipeline
# Workspace : Daily_Sales_Analytics
# Lakehouse  : Pulse_Sales_Silver  (attach as default lakehouse)
# Trigger    : Automatically after Bronze pipeline Copy Data activity succeeds
#
# ─────────────────────────────────────────────────────────────────────────────
# HOW TO IMPORT INTO FABRIC
# 1. Open your Fabric workspace → New → Import notebook → upload this file
# 2. Attach Pulse_Sales_Silver as the default lakehouse
# 3. Also add Pulse_Sales_Bronze as a secondary lakehouse (so audit is readable)
# ─────────────────────────────────────────────────────────────────────────────

# ── Notebook Cell 1 ──────────────────────────────────────────────────────────
# TITLE: Silver Layer — Sales Pipeline
# Medallion layer : Silver
# Source          : Pulse_Sales_Bronze  → Files/bronze/sales/*.csv
# Destination     : Pulse_Sales_Silver  → Tables/sales_silver (Delta)
# Supporting tbls : silver_watermark, silver_audit_log
# ─────────────────────────────────────────────────────────────────────────────


StatementMeta(, 7cb4d051-bb53-45bf-9cc6-b29b4baa9c40, 3, Finished, Available, Finished, False)

In [2]:

# ── Notebook Cell 2 ──────────────────────────────────────────────────────────
# TITLE: Imports & configuration

import uuid
import json
import time
import re
from datetime import datetime

from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DateType, IntegerType, DecimalType,
    TimestampType, FloatType
)
from delta.tables import DeltaTable

# ── Path constants ────────────────────────────────────────────────────────────
#BRONZE_FILES_PATH      = "Files/bronze/sales/"          # CSV source files
BRONZE_FILES_PATH      = "abfss://cfe97b73-8a76-467c-8253-9e4fe85794ea@onelake.dfs.fabric.microsoft.com/643120cf-e905-435a-8b05-c7de3ff1d062/Files/bronze/sales/"
SILVER_TABLE_PATH      = "Tables/sales_silver"          # main Delta output
WATERMARK_TABLE_PATH   = "Tables/silver_watermark"      # processed-file tracker
AUDIT_TABLE_PATH       = "Tables/silver_audit_log"      # per-run audit rows

SILVER_TABLE_NAME      = "sales_silver"
WATERMARK_TABLE_NAME   = "silver_watermark"
AUDIT_TABLE_NAME       = "silver_audit_log"

#lakehouse name
BRONZE_LAKEHOUSE       = "Pulse_Sales_Bronze"
SILVER_LAKEHOUSE       = "Pulse_Sales_Silver"

print("Configuration loaded.")
print(f"  Bronze source : {BRONZE_FILES_PATH}")
print(f"  Silver target : {SILVER_TABLE_PATH}")



StatementMeta(, 7cb4d051-bb53-45bf-9cc6-b29b4baa9c40, 4, Finished, Available, Finished, False)

Configuration loaded.
  Bronze source : abfss://cfe97b73-8a76-467c-8253-9e4fe85794ea@onelake.dfs.fabric.microsoft.com/643120cf-e905-435a-8b05-c7de3ff1d062/Files/bronze/sales/
  Silver target : Tables/sales_silver


In [3]:

# ── Notebook Cell 3 ──────────────────────────────────────────────────────────
# TITLE: Schema definitions

# Source CSV schema (enforced on read — rejects structurally malformed rows)
SOURCE_SCHEMA = StructType([
    StructField("TransactionID", StringType(),       nullable=True),
    StructField("Date",          DateType(),         nullable=True),
    StructField("StoreID",       StringType(),       nullable=True),
    StructField("ProductID",     StringType(),       nullable=True),
    StructField("Quantity",      IntegerType(),      nullable=True),
    StructField("UnitPrice",     DecimalType(10, 2), nullable=True),
    StructField("CustomerEmail", StringType(),       nullable=True),
])

# silver_watermark schema
WATERMARK_SCHEMA = StructType([
    StructField("file_name",      StringType(),   nullable=False),
    StructField("file_date",      StringType(),   nullable=True),   # YYYYMMDD string
    StructField("silver_status",  StringType(),   nullable=False),  # PROCESSED / FAILED
    StructField("processed_at",   TimestampType(),nullable=False),
    StructField("rows_written",   IntegerType(),  nullable=True),
])

# silver_audit_log schema
AUDIT_SCHEMA = StructType([
    StructField("run_id",             StringType(),   nullable=False),
    StructField("file_name",          StringType(),   nullable=False),
    StructField("processed_at",       TimestampType(),nullable=False),
    StructField("rows_read",          IntegerType(),  nullable=True),
    StructField("rows_rejected",      IntegerType(),  nullable=True),
    StructField("rows_written",       IntegerType(),  nullable=True),
    StructField("rejection_reasons",  StringType(),   nullable=True),  # JSON string
    StructField("silver_status",      StringType(),   nullable=False),  # SUCCESS / FAILED
    StructField("duration_seconds",   FloatType(),    nullable=True),
])

print("Schemas defined.")



StatementMeta(, 7cb4d051-bb53-45bf-9cc6-b29b4baa9c40, 5, Finished, Available, Finished, False)

Schemas defined.


In [4]:

# ── Notebook Cell 4 ──────────────────────────────────────────────────────────
# TITLE: Helper — initialise supporting tables on first run

def ensure_watermark_table_exists():
    """
    Creates silver_watermark as an empty Delta table if it does not
    already exist.  Safe to call on every run.
    """
    if not DeltaTable.isDeltaTable(spark, WATERMARK_TABLE_PATH):
        empty_df = spark.createDataFrame([], WATERMARK_SCHEMA)
        (empty_df.write
            .format("delta")
            .mode("overwrite")
            .option("path", WATERMARK_TABLE_PATH)
            .saveAsTable(f"{SILVER_LAKEHOUSE}.{WATERMARK_TABLE_NAME}"))
        print("  Created silver_watermark table.")
    else:
        print("  silver_watermark already exists — skipping creation.")


def ensure_audit_table_exists():
    """
    Creates silver_audit_log as an empty Delta table if it does not
    already exist.  Safe to call on every run.
    """
    if not DeltaTable.isDeltaTable(spark, AUDIT_TABLE_PATH):
        empty_df = spark.createDataFrame([], AUDIT_SCHEMA)
        (empty_df.write
            .format("delta")
            .mode("overwrite")
            .option("path", AUDIT_TABLE_PATH)
            .saveAsTable(f"{SILVER_LAKEHOUSE}.{AUDIT_TABLE_NAME}"))
        print("  Created silver_audit_log table.")
    else:
        print("  silver_audit_log already exists — skipping creation.")


ensure_watermark_table_exists()
ensure_audit_table_exists()



StatementMeta(, 7cb4d051-bb53-45bf-9cc6-b29b4baa9c40, 6, Finished, Available, Finished, False)

  silver_watermark already exists — skipping creation.
  silver_audit_log already exists — skipping creation.


In [5]:

# ── Notebook Cell 5 ──────────────────────────────────────────────────────────
# TITLE: Step 1 — Identify pending files (cross-reference bronze audit × silver watermark)
#
# Logic:
#   pending = files in bronze_audit_log that are either:
#     (a) not present in silver_watermark at all  → never processed
#     (b) present but bronze_audit_log.loaded_at is newer than
#         silver_watermark.processed_at           → re-landed at source, needs reprocessing

bronze_audit_log_df = spark.sql(f"""
    SELECT
        FileName        AS file_name,
        DataLoadedDate  AS bronze_loaded_at
    FROM {BRONZE_LAKEHOUSE}.bronze_audit_log
""")
silver_watermark_df = spark.sql(f"""
    SELECT
        file_name,
        processed_at    AS silver_processed_at,
        silver_status
    FROM {SILVER_LAKEHOUSE}.{WATERMARK_TABLE_NAME}
    WHERE silver_status = 'PROCESSED'
""")

pending_df = (
    bronze_audit_log_df
    .join(silver_watermark_df, on="file_name", how="left")
    .filter(
        F.col("silver_processed_at").isNull() |                          # (a) never processed
        (F.col("bronze_loaded_at") > F.col("silver_processed_at"))       # (b) re-landed
    )
    .select("file_name", "bronze_loaded_at")
    .orderBy("file_name")   # oldest date first so silver is built chronologically
)

pending_files = pending_df.collect()

print(f"\nPending files to process: {len(pending_files)}")
for f in pending_files:
    print(f"   {f.file_name}  (bronze loaded: {f.bronze_loaded_at})")

if not pending_files:
    print("\nNothing to process — silver is up to date. Exiting.")
    mssparkutils.notebook.exit("NO_PENDING_FILES")


StatementMeta(, 7cb4d051-bb53-45bf-9cc6-b29b4baa9c40, 7, Finished, Available, Finished, False)


Pending files to process: 4
   sales_20260301.csv  (bronze loaded: 2026-03-26 21:09:29)
   sales_20260301.csv  (bronze loaded: 2026-03-26 21:15:46)
   sales_20260302.csv  (bronze loaded: 2026-03-26 21:19:21)
   sales_20260303.csv  (bronze loaded: 2026-03-26 21:21:14)


In [6]:


# ── Notebook Cell 6 ──────────────────────────────────────────────────────────
# TITLE: Steps 2-6 — Main processing loop
#
# For each pending file:
#   2. Read CSV from Bronze with enforced schema
#   3. Cleanse  — drop null TransactionID, nulls in required fields,
#                 invalid qty/price, duplicates, trim whitespace
#   4. Enrich   — derived columns, PII masking, audit columns
#   5. Load     — DELETE existing rows for this file + INSERT fresh rows
#                 (atomic via Delta transaction log; handles file shrinkage cleanly)
#   6. Watermark + audit update

for file_row in pending_files:

    file_name   = file_row.file_name
    source_path = f"{BRONZE_FILES_PATH}{file_name}"
    run_id      = str(uuid.uuid4())
    run_start   = time.time()
    rejection_reasons = {}

    print(f"\n{'─'*60}")
    print(f"Processing : {file_name}")
    print(f"Run ID     : {run_id}")
    print(f"{'─'*60}")

    try:

        # ── STEP 2: Read with enforced schema ─────────────────────────
        df_raw = (spark.read
                  .option("header",          True)
                  .option("mode",            "PERMISSIVE")   # don't abort on bad rows
                  .option("dateFormat",      "yyyy-MM-dd")
                  .schema(SOURCE_SCHEMA)
                  .csv(source_path))

        rows_read = df_raw.cache().count()
        print(f"  Rows read         : {rows_read}")


        # ── STEP 3: Cleanse ────────────────────────────────────────────

        df = df_raw

        # Rule 1 — Drop rows where TransactionID is null
        null_tid_count = df.filter(F.col("TransactionID").isNull()).count()
        if null_tid_count:
            rejection_reasons["null_transaction_id"] = null_tid_count
        df = df.filter(F.col("TransactionID").isNotNull())

        # Rule 2 — Trim whitespace on all string columns
        string_cols = [
            f.name for f in SOURCE_SCHEMA.fields
            if isinstance(f.dataType, StringType)
        ]
        for col_name in string_cols:
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

        # Rule 3 — Drop rows with null in other required fields
        before = df.count()
        df = df.dropna(subset=["Date", "Quantity", "UnitPrice"])
        null_required_count = before - df.count()
        if null_required_count:
            rejection_reasons["null_required_fields"] = null_required_count

        # Rule 4 — Drop rows with zero or negative Quantity / UnitPrice
        before = df.count()
        df = df.filter((F.col("Quantity") > 0) & (F.col("UnitPrice") > 0))
        bad_values_count = before - df.count()
        if bad_values_count:
            rejection_reasons["invalid_qty_or_price"] = bad_values_count

        # Rule 5 — Deduplicate on TransactionID, keep first occurrence
        before = df.count()
        df = df.dropDuplicates(["TransactionID"])
        dup_count = before - df.count()
        if dup_count:
            rejection_reasons["duplicate_transaction_id"] = dup_count

        rows_rejected = rows_read - df.count()
        print(f"  Rows rejected     : {rows_rejected}")
        if rejection_reasons:
            print(f"  Rejection detail  : {rejection_reasons}")


        # ── STEP 4: Enrich ─────────────────────────────────────────────

        # Extract date part from filename for traceability
        file_date_match = re.search(r'(\d{8})', file_name)
        file_date_str   = file_date_match.group(1) if file_date_match else "00000000"

        df_enriched = (df
            # Business metric
            .withColumn("TotalRevenue",
                        (F.col("Quantity") * F.col("UnitPrice"))
                        .cast(DecimalType(12, 2)))

            # Date dimension columns (useful for partitioning & Gold aggregations)
            .withColumn("Year",  F.year("Date"))
            .withColumn("Month", F.month("Date"))
            .withColumn("Day",   F.dayofmonth("Date"))

            # PII: extract domain first, then mask the full address
            .withColumn("EmailDomain",
                        F.when(
                            F.col("CustomerEmail").isNotNull() &
                            F.col("CustomerEmail").contains("@"),
                            F.split("CustomerEmail", "@").getItem(1)
                        ).otherwise(F.lit(None).cast(StringType())))

            .withColumn("CustomerEmail",
                        F.when(
                            F.col("CustomerEmail").isNotNull() &
                            F.col("CustomerEmail").contains("@"),
                            F.concat(F.lit("***@"),
                                     F.split("CustomerEmail", "@").getItem(1))
                        ).otherwise(F.lit("***")))

            # Row-level audit columns
            .withColumn("_source_file",      F.lit(file_name))
            .withColumn("_silver_loaded_at", F.current_timestamp())
            .withColumn("_silver_run_id",    F.lit(run_id))
        )

        rows_written = df_enriched.count()
        print(f"  Rows to write     : {rows_written}")


        # ── STEP 5: Load into sales_silver (DELETE + INSERT) ───────────
        #
        # Strategy: DELETE all rows previously loaded from this specific file,
        # then INSERT the complete fresh set.
        #
     

        if not DeltaTable.isDeltaTable(spark, SILVER_TABLE_PATH):
            # First ever run — create the table
            (df_enriched.write
                .format("delta")
                .partitionBy("Year", "Month")
                .mode("overwrite")
                .option("path", SILVER_TABLE_PATH)
                .saveAsTable(f"{SILVER_LAKEHOUSE}.{SILVER_TABLE_NAME}"))
            print(f"  Created sales_silver and wrote {rows_written} rows.")

        else:
            silver_table = DeltaTable.forPath(spark, SILVER_TABLE_PATH)

            # Step 5a — remove all rows from this source file (handles deletions)
            silver_table.delete(F.col("_source_file") == F.lit(file_name))

            # Step 5b — insert the complete fresh set
            (df_enriched.write
                .format("delta")
                .mode("append")
                .option("path", SILVER_TABLE_PATH)
                .save())

            print(f"  Deleted old rows for {file_name}, inserted {rows_written} fresh rows.")


        # ── STEP 6a: Update silver_watermark ──────────────────────────

        watermark_row = spark.createDataFrame(
            [{
                "file_name":     file_name,
                "file_date":     file_date_str,
                "silver_status": "PROCESSED",
                "processed_at":  datetime.now(),
                "rows_written":  rows_written,
            }],
            schema=WATERMARK_SCHEMA
        )

        wm_table = DeltaTable.forPath(spark, WATERMARK_TABLE_PATH)
        (wm_table.alias("wm")
            .merge(watermark_row.alias("new"), "wm.file_name = new.file_name")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())

        print(f"  Watermark updated.")


        # ── STEP 6b: Write silver_audit_log row ────────────────────────

        duration = round(time.time() - run_start, 2)

        audit_row = spark.createDataFrame(
            [{
                "run_id":            run_id,
                "file_name":         file_name,
                "processed_at":      datetime.now(),
                "rows_read":         rows_read,
                "rows_rejected":     rows_rejected,
                "rows_written":      rows_written,
                "rejection_reasons": json.dumps(rejection_reasons),
                "silver_status":     "SUCCESS",
                "duration_seconds":  duration,
            }],
            schema=AUDIT_SCHEMA
        )

        audit_row.write.format("delta").mode("append").saveAsTable(
            f"{SILVER_LAKEHOUSE}.{AUDIT_TABLE_NAME}"
        )

        print(f"  Audit row written.")
        print(f"  {file_name} complete — {rows_written} rows, {rows_rejected} rejected, {duration}s")


    except Exception as e:
        # ── Error handling — log failure to audit + watermark then continue ──
        duration = round(time.time() - run_start, 2)
        error_msg = str(e)
        print(f"\n  ✗ FAILED: {file_name}")
        print(f"    Error: {error_msg}")

        # Write FAILED status to watermark so we can retry next run
        failed_wm_row = spark.createDataFrame(
            [{
                "file_name":     file_name,
                "file_date":     re.search(r'(\d{8})', file_name).group(1) if re.search(r'(\d{8})', file_name) else "00000000",
                "silver_status": "FAILED",
                "processed_at":  datetime.now(),
                "rows_written":  0,
            }],
            schema=WATERMARK_SCHEMA
        )
        wm_table = DeltaTable.forPath(spark, WATERMARK_TABLE_PATH)
        (wm_table.alias("wm")
            .merge(failed_wm_row.alias("new"), "wm.file_name = new.file_name")
            .whenMatchedUpdateAll()
            .whenNotMatchedInsertAll()
            .execute())

        # Write FAILED audit row
        failed_audit_row = spark.createDataFrame(
            [{
                "run_id":            run_id,
                "file_name":         file_name,
                "processed_at":      datetime.now(),
                "rows_read":         0,
                "rows_rejected":     0,
                "rows_written":      0,
                "rejection_reasons": json.dumps({"error": error_msg}),
                "silver_status":     "FAILED",
                "duration_seconds":  duration,
            }],
            schema=AUDIT_SCHEMA
        )
        failed_audit_row.write.format("delta").mode("append").saveAsTable(
            f"{SILVER_LAKEHOUSE}.{AUDIT_TABLE_NAME}"
        )

        # Continue to next file — do not abort the entire run
        continue



StatementMeta(, 7cb4d051-bb53-45bf-9cc6-b29b4baa9c40, 8, Finished, Available, Finished, False)


────────────────────────────────────────────────────────────
Processing : sales_20260301.csv
Run ID     : 9eabc54a-1e3f-4735-9d52-f9845c03ea39
────────────────────────────────────────────────────────────
  Rows read         : 4
  Rows rejected     : 0
  Rows to write     : 4
  Deleted old rows for sales_20260301.csv, inserted 4 fresh rows.
  Watermark updated.
  Audit row written.
  sales_20260301.csv complete — 4 rows, 0 rejected, 23.96s

────────────────────────────────────────────────────────────
Processing : sales_20260301.csv
Run ID     : 09e655a8-ff8a-43bd-a0f2-3a54dc62a264
────────────────────────────────────────────────────────────
  Rows read         : 4
  Rows rejected     : 0
  Rows to write     : 4
  Deleted old rows for sales_20260301.csv, inserted 4 fresh rows.
  Watermark updated.
  Audit row written.
  sales_20260301.csv complete — 4 rows, 0 rejected, 18.83s

────────────────────────────────────────────────────────────
Processing : sales_20260302.csv
Run ID     : c3fc4

In [7]:

# ── Notebook Cell 7 ──────────────────────────────────────────────────────────
# TITLE: Run summary

print(f"\n{'═'*60}")
print("SILVER LAYER RUN COMPLETE")
print(f"{'═'*60}")

summary = spark.sql(f"""
    SELECT
        silver_status,
        COUNT(*)            AS files,
        SUM(rows_written)   AS total_rows_written,
        ROUND(SUM(duration_seconds), 1) AS total_seconds
    FROM {SILVER_LAKEHOUSE}.{AUDIT_TABLE_NAME}
    WHERE run_id IN (
        SELECT DISTINCT run_id
        FROM {SILVER_LAKEHOUSE}.{AUDIT_TABLE_NAME}
        WHERE DATE(processed_at) = CURRENT_DATE()
    )
    GROUP BY silver_status
""")

summary.show(truncate=False)

print("\nWatermark state:")
spark.sql(f"""
    SELECT file_name, silver_status, processed_at, rows_written
    FROM {SILVER_LAKEHOUSE}.{WATERMARK_TABLE_NAME}
    ORDER BY file_name
""").show(truncate=False)

StatementMeta(, 7cb4d051-bb53-45bf-9cc6-b29b4baa9c40, 9, Finished, Available, Finished, False)


════════════════════════════════════════════════════════════
SILVER LAYER RUN COMPLETE
════════════════════════════════════════════════════════════
+-------------+-----+------------------+-------------+
|silver_status|files|total_rows_written|total_seconds|
+-------------+-----+------------------+-------------+
|SUCCESS      |4    |18                |75.4         |
+-------------+-----+------------------+-------------+


Watermark state:
+------------------+-------------+--------------------------+------------+
|file_name         |silver_status|processed_at              |rows_written|
+------------------+-------------+--------------------------+------------+
|sales_20260301.csv|PROCESSED    |2026-03-30 17:04:42.239048|4           |
|sales_20260302.csv|PROCESSED    |2026-03-30 17:05:03.810473|5           |
|sales_20260303.csv|PROCESSED    |2026-03-30 17:05:23.498855|5           |
+------------------+-------------+--------------------------+------------+

